In [0]:
# ============================================================
# KAFKA CONNECTION TEST
# ============================================================

dbutils.widgets.text(
    "kafka_bootstrap",
    "pkc-921jm.us-east-2.aws.confluent.cloud:9092",
    "Kafka Bootstrap Server"
)

dbutils.widgets.text(
    "kafka_api_key",
    "",
    "Kafka API Key"
)

dbutils.widgets.text(
    "kafka_api_secret",
    "",
    "Kafka API Secret"
)

dbutils.widgets.text(
    "kafka_topic",
    "orders",
    "Kafka Topic"
)

bootstrap = dbutils.widgets.get("kafka_bootstrap")
api_key = dbutils.widgets.get("kafka_api_key")
api_secret = dbutils.widgets.get("kafka_api_secret")
topic = dbutils.widgets.get("kafka_topic")

print("Bootstrap:", bootstrap)
print("API Key:", api_key)
print("Topic:", topic)
print("Secret loaded:", bool(api_secret))

In [0]:
# ============================================================
# KAFKA → DATABRICKS STRUCTURED STREAMING
# ============================================================

from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)

# Read current widget values
bootstrap = dbutils.widgets.get("kafka_bootstrap")
api_key = dbutils.widgets.get("kafka_api_key")
api_secret = dbutils.widgets.get("kafka_api_secret")
topic = dbutils.widgets.get("kafka_topic")

# JSON schema for Kafka order events
order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("order_time", StringType(), True)
])

# Kafka connection configuration
kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "subscribe": topic,
    "startingOffsets": "earliest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
        f'required username="{api_key}" password="{api_secret}";'
    )
}

# Read from Kafka
kafka_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

# Parse JSON values
orders_stream = (
    kafka_df
    .select(
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp"),
        col("value").cast("string").alias("json_value")
    )
    .withColumn("order", from_json(col("json_value"), order_schema))
    .select(
        "partition",
        "offset",
        "kafka_timestamp",
        "order.*"
    )
    .withColumn(
        "order_time",
        to_timestamp("order_time")
    )
)

# Delta output location
output_path = "/Volumes/workspace/lakehouse_pipeline/raw_data/streaming/orders"

checkpoint_path = "/Volumes/workspace/lakehouse_pipeline/raw_data/checkpoints/kafka_orders"

# Process all currently available Kafka messages
query = (
    orders_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(output_path)
)

query.awaitTermination()

print("============================================")
print("KAFKA STREAMING COMPLETED")
print("============================================")

streaming_orders = spark.read.format("delta").load(output_path)

print("Records received:", streaming_orders.count())

display(
    streaming_orders
    .orderBy("offset")
)